# 07 — Build institutional-ownership (IO) panel

Pulls quarterly 13F institutional holdings from WRDS and computes the annual
institutional-ownership ratio (`io` = institutional shares held / shares outstanding)
for every GVKEY in `../Returns/gvkeys.txt`.

**Output**: `../data/processed/io_panel.parquet` with columns `gvkey`, `fyear`, `io`, `io_min`, `io_max`, `n_q`.

**Caveat.** Thomson Reuters 13F (`tfn.s34type3`) coverage ends around mid-2020 on
most WRDS subscriptions. For post-2020 data swap in the Refinitiv-13F replacement
(`wrdsapps.tr_13f_holding` or similar) or FactSet ownership — the query structure is identical.
Pre-2020 coverage is the most useful piece for the γ_pre / β-channel test anyway.

In [1]:
import pandas as pd
import numpy as np
import wrds

from pathlib import Path

In [2]:
GVKEY_FILE  = "../Returns/gvkeys.txt"
DATE_CUTOFF = "2010-01-01"

OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
db = wrds.Connection(wrds_username="gcrippa4")

Loading library list...
Done


## GVKEY universe → SQL list

In [4]:
gvkeys = (
    pd.read_csv(GVKEY_FILE, header=None)[0]
      .astype(str).str.strip().str.zfill(6)
      .tolist()
)
gvlist_sql = ",".join("'" + g + "'" for g in gvkeys)
print(f"{len(gvkeys):,} GVKEYs loaded")

13,303 GVKEYs loaded


## GVKEY → PERMNO (CCM link table)

In [5]:
ccm = db.raw_sql(f"""
    select gvkey, lpermno as permno, linktype, linkprim, linkdt, linkenddt
    from crsp.ccmxpf_linktable
    where gvkey in ({gvlist_sql})
      and linktype in ('LU','LC','LN','LS','LX','LD')
      and linkprim in ('P','C')
""", date_cols=['linkdt','linkenddt'])

ccm['linkenddt'] = ccm['linkenddt'].fillna(pd.Timestamp("2099-12-31"))
ccm = ccm.dropna(subset=['permno']).copy()
ccm['permno'] = ccm['permno'].astype(int)
print(f"{ccm['gvkey'].nunique():,} GVKEYs matched to {ccm['permno'].nunique():,} PERMNOs")

9,447 GVKEYs matched to 9,852 PERMNOs


## PERMNO → historical 8-digit CUSIP (CRSP `stocknames`)

13F uses **8-digit CUSIPs**, and CUSIPs change over a firm's life (spin-offs, restructurings),
so we keep every `(permno, ncusip, date-range)` tuple and do a date-bounded merge later.

In [6]:
permnos    = ccm['permno'].unique()
permno_sql = ",".join(str(int(p)) for p in permnos)

names = db.raw_sql(f"""
    select permno, ncusip, namedt, nameenddt
    from crsp.stocknames
    where permno in ({permno_sql})
      and ncusip is not null
""", date_cols=['namedt','nameenddt'])

names['nameenddt'] = names['nameenddt'].fillna(pd.Timestamp("2099-12-31"))
names['ncusip8']   = names['ncusip'].str[:8]
cusips = names['ncusip8'].dropna().unique()
print(f"{len(cusips):,} unique 8-digit CUSIPs")

16,349 unique 8-digit CUSIPs


## Quarterly 13F holdings (aggregated server-side)

Server-side `sum(shares)` over `mgrno` collapses millions of manager-level rows
into one row per `(cusip, rdate)` before transfer.

> If your subscription has switched to post-Refinitiv 13F, swap `tfn.s34type3`
> for the equivalent table (e.g. `wrdsapps.tr_13f_holding`). Columns are the same.

In [7]:
cusip_sql = ",".join("'" + c + "'" for c in cusips)

holdings = db.raw_sql(f"""
    select rdate, cusip, sum(shares) as inst_shares
    from tfn.s34
    where rdate >= '{DATE_CUTOFF}'
      and cusip in ({cusip_sql})
    group by rdate, cusip
""", date_cols=['rdate'])

print(
    f"{len(holdings):,} (cusip, quarter) rows; coverage "
    f"{holdings['rdate'].min().date()} → {holdings['rdate'].max().date()}"
)

335,743 (cusip, quarter) rows; coverage 2010-03-31 → 2025-09-30


## Quarterly shares outstanding from CRSP `msf`

CRSP `shrout` is in thousands of shares. `crsp.msf` is monthly; 13F `rdate` is
the quarter-end. We match on year-month so the IO denominator is the actual
shares outstanding *at the report date*.

In [8]:
msf = db.raw_sql(f"""
    select permno, date as mdate, shrout
    from crsp.msf
    where date >= '{DATE_CUTOFF}'
      and permno in ({permno_sql})
""", date_cols=['mdate'])

msf['shares_out'] = msf['shrout'].astype(float) * 1000.0
msf['ym']         = msf['mdate'].dt.to_period('M')
print(f"{len(msf):,} CRSP monthly obs")

899,835 CRSP monthly obs


## Merge `(permno, ncusip, date)` → 13F quarter → CRSP shrout → quarterly IO

In [9]:
# (cusip, rdate) holdings → attach permno via date-bounded ncusip match
holdings['ym'] = holdings['rdate'].dt.to_period('M')
nm = names[['permno','ncusip8','namedt','nameenddt']]
hp = holdings.merge(nm, left_on='cusip', right_on='ncusip8')
hp = hp[(hp['rdate'] >= hp['namedt']) & (hp['rdate'] <= hp['nameenddt'])]
hp = hp.drop_duplicates(['permno','rdate'])

# attach shrout at the same month-end
hp = hp.merge(msf[['permno','ym','shares_out']], on=['permno','ym'], how='left')
hp = hp.dropna(subset=['shares_out'])
hp = hp[hp['shares_out'] > 0].copy()

# quarterly IO  — cap at 1.5 (values >1 happen with short positions / stale data)
hp['io_qtr'] = (hp['inst_shares'] / hp['shares_out']).clip(upper=1.5)
print(hp[['io_qtr']].describe().round(3))

         io_qtr
count  279741.0
mean      0.572
std       0.345
min         0.0
25%       0.257
50%       0.635
75%       0.867
max         1.5


## Aggregate to fiscal year and merge GVKEY

Average across the four quarter-ends per calendar year, then re-attach `gvkey`
via the CCM link (date-bounded).

In [10]:
hp['fyear'] = hp['rdate'].dt.year

io_annual = (
    hp.groupby(['permno','fyear'])['io_qtr']
      .agg(io='mean', io_min='min', io_max='max', n_q='count')
      .reset_index()
)

io_annual['ydate'] = pd.to_datetime(io_annual['fyear'].astype(str) + '-06-30')
ic = io_annual.merge(ccm[['gvkey','permno','linkdt','linkenddt']], on='permno')
ic = ic[(ic['ydate'] >= ic['linkdt']) & (ic['ydate'] <= ic['linkenddt'])]
io_panel = (
    ic[['gvkey','fyear','io','io_min','io_max','n_q']]
      .drop_duplicates(['gvkey','fyear'])
      .reset_index(drop=True)
)

print(f"{len(io_panel):,} firm-year IO observations")
print(io_panel.describe().round(3))

71,035 firm-year IO observations
           fyear       io   io_min   io_max      n_q
count  71035.000  71035.0  71035.0  71035.0  71035.0
mean    2017.288    0.569    0.533    0.608    3.843
std        4.348    0.342     0.34    0.353    0.521
min     2010.000      0.0      0.0      0.0      1.0
25%     2014.000    0.256     0.21    0.294      4.0
50%     2017.000     0.63    0.588    0.676      4.0
75%     2021.000    0.866    0.834      0.9      4.0
max     2024.000      1.5      1.5      1.5      4.0


## Save

In [11]:
out = OUT_DIR / "io_panel.parquet"
io_panel.to_parquet(out, index=False)
print(f"Saved → {out}")

Saved → ../data/processed/io_panel.parquet


In [12]:
io_panel

,gvkey,fyear,io,io_min,io_max,n_q
0,012994,2010,0.100045,0.075604,0.124283,3
1,012994,2011,0.087724,0.075282,0.108631,4
2,012994,2012,0.066518,0.05149,0.081454,4
3,012994,2013,0.085966,0.050701,0.113955,4
4,012994,2014,0.164449,0.114996,0.21644,4
...,...,...,...,...,...,...
71030,184996,2020,0.46415,0.419861,0.499342,4
71031,184996,2021,0.407946,0.39892,0.415327,4
71032,184996,2022,0.423194,0.411398,0.428524,4
71033,184996,2023,0.418327,0.408904,0.433566,4
